# Day 20 — Frameworks: LangChain vs LlamaIndex

LangChain and LlamaIndex package the pieces you've already built by hand — prompts, tools,
retrievers, agent loops, memory. This hour: build tiny versions of each framework's core
abstraction so the real APIs stop being magic, map your Week 6–7 code onto them, and learn how
to choose.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | What a framework gives you (and takes) | 4 min |
| 1 | LangChain's core idea: composable `Runnable`s (LCEL) | 14 min |
| 2 | LangChain agents + tools + memory | 10 min |
| 3 | LlamaIndex's core idea: index → query engine | 14 min |
| 4 | The real quickstarts, side by side | 10 min |
| 5 | Choosing: LangChain vs LlamaIndex vs neither | 5 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import numpy as np, re, json
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("all-MiniLM-L6-v2")
print("ready")

/Users/umeshkaranam/Desktop/personal/UPSKILL/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5078.86it/s]

ready


## 0 — What a framework gives you (4 min)

**Gives:** ready-made components (100+ vector-store integrations, doc loaders, memory types,
agent loops, output parsers), a composition syntax, streaming/callbacks/tracing hooks, and a
community of examples.

**Takes:** a dependency tree and its churn, an abstraction layer to learn and debug through,
and opinions that fight you when your use case is off the beaten path.

Rule of thumb: **build the first version yourself** (you now can — Weeks 5–7), reach for a
framework when you're spending real time reimplementing loaders/integrations/streaming, or
when a team needs a shared vocabulary.

## 1 — LangChain's core idea: `Runnable` composition (14 min)

LangChain's spine is **LCEL** (LangChain Expression Language): every component is a `Runnable`
with an `.invoke()` method, and you pipe them with `|`. `prompt | llm | parser` is a chain.

We'll build a 20-line `Runnable` that supports `|`.

In [2]:
class Runnable:
    def __init__(self, fn): self.fn = fn
    def invoke(self, x): return self.fn(x)
    def __or__(self, other):                      # enables:  a | b | c
        other = other if isinstance(other, Runnable) else Runnable(other)
        return Runnable(lambda x: other.invoke(self.invoke(x)))
    def batch(self, xs): return [self.invoke(x) for x in xs]

# --- components, LangChain-style ---
def PromptTemplate(template):
    return Runnable(lambda vars: template.format(**vars))

def FakeLLM(system="You are helpful."):
    # stand-in for ChatAnthropic(model="claude-opus-5")
    def call(prompt):
        if "sentiment" in prompt.lower():
            return "positive" if any(w in prompt.lower() for w in ["love","great","good"]) else "negative"
        if "json" in prompt.lower():
            return '{"answer": "42"}'
        return f"[LLM answer to: {prompt[:40]}...]"
    return Runnable(call)

def StrOutputParser(): return Runnable(lambda s: s.strip())
def JsonOutputParser(): return Runnable(lambda s: json.loads(re.search(r"\{.*\}", s, re.S).group()))

# a chain, exactly like:  prompt | llm | parser
chain = PromptTemplate("Classify the sentiment of: {text}") | FakeLLM() | StrOutputParser()
print(chain.invoke({"text": "I love this product"}))
print(chain.batch([{"text": "great value"}, {"text": "broke on day one"}]))

json_chain = PromptTemplate("Answer as JSON: {q}") | FakeLLM() | JsonOutputParser()
print(json_chain.invoke({"q": "what is 6*7"}))

positive
['positive', 'negative']
{'answer': '42'}


That's the whole mental model. Real LangChain adds: async (`ainvoke`), streaming (`stream`),
`RunnableParallel` / `RunnablePassthrough` for branching, automatic retries, and
`.with_config()` for callbacks/tracing (LangSmith). But `component | component | component`
is the idea, and it's why LCEL chains are easy to read and swap.

In [3]:
# RunnableParallel: run several sub-chains on the same input, collect a dict
class RunnableParallel(Runnable):
    def __init__(self, **branches): self.branches = branches
    def invoke(self, x): return {k: v.invoke(x) for k, v in self.branches.items()}

# a RAG-shaped chain: retrieve + passthrough question -> prompt -> llm
DOCS = ["Refunds take 5 business days.", "Shipping is free over $50.",
        "Support hours are 9 to 5 Pacific.", "The API limit is 600 requests per minute."]
DVEC = emb.encode(DOCS, normalize_embeddings=True)
retriever = Runnable(lambda q: DOCS[int(np.argmax(DVEC @ emb.encode([q], normalize_embeddings=True)[0]))])

rag_chain = (
    RunnableParallel(context=retriever, question=Runnable(lambda q: q))
    | Runnable(lambda d: f"Context: {d['context']}\nQ: {d['question']}\nA:")
    | FakeLLM()
    | StrOutputParser()
)
print(rag_chain.invoke("how long do refunds take"))

[LLM answer to: Context: Refunds take 5 business days.
Q...]


## 2 — LangChain agents + tools + memory (10 min)

LangChain wraps the Day 19 loop as an `AgentExecutor` over `Tool` objects, plus `Memory` that
auto-injects conversation history into the prompt.

In [4]:
class Tool:
    def __init__(self, name, description, func): self.name, self.description, self.func = name, description, func

class ConversationBufferMemory:
    def __init__(self): self.messages = []
    def load(self): return "\n".join(f"{r}: {c}" for r, c in self.messages)
    def save(self, user, ai): self.messages += [("Human", user), ("AI", ai)]

class AgentExecutor:
    def __init__(self, tools, llm_planner, memory=None, max_iterations=6):
        self.tools = {t.name: t for t in tools}
        self.planner = llm_planner; self.memory = memory; self.max_iterations = max_iterations
    def invoke(self, user_input):
        scratch = []
        for i in range(self.max_iterations):
            step = self.planner(user_input, scratch, self.memory.load() if self.memory else "")
            if step["type"] == "final":
                if self.memory: self.memory.save(user_input, step["output"])
                return step["output"]
            obs = self.tools[step["tool"]].func(step["input"]) if step["tool"] in self.tools else "unknown tool"
            scratch.append((step["tool"], step["input"], obs))
        return "stopped: max_iterations"

WORD_LENGTHS = Tool("word_count", "Count words in a string.", lambda s: len(s.split()))
UPPER = Tool("uppercase", "Uppercase a string.", lambda s: s.upper())

def planner(user, scratch, history):
    if not scratch:
        return {"type": "action", "tool": "word_count", "input": user}
    if len(scratch) == 1:
        return {"type": "action", "tool": "uppercase", "input": user}
    return {"type": "final", "output": f"{scratch[0][2]} words; shouted: {scratch[1][2]}"}

agent = AgentExecutor([WORD_LENGTHS, UPPER], planner, memory=ConversationBufferMemory())
print(agent.invoke("the quick brown fox"))
print("memory now:", agent.memory.messages)

4 words; shouted: THE QUICK BROWN FOX
memory now: [('Human', 'the quick brown fox'), ('AI', '4 words; shouted: THE QUICK BROWN FOX')]


Real `AgentExecutor` uses the model's native tool-calling (not our string planner), supports
`create_tool_calling_agent`, and integrates `@tool`-decorated functions. LangGraph is the
newer, lower-level successor for agents that need explicit state machines, branching, and
human-in-the-loop checkpoints — LangChain now recommends LangGraph for anything beyond a
simple loop.

## 3 — LlamaIndex's core idea: index → query engine (14 min)

LlamaIndex is **retrieval-first**. The central objects: `Document` → (node parser) → `Node`s →
`VectorStoreIndex` → `.as_query_engine()` → `.query()` returns an answer *with sources*. It
hides the chunk/embed/retrieve/synthesize pipeline behind two calls.

In [5]:
class Document:
    def __init__(self, text, metadata=None): self.text = text; self.metadata = metadata or {}

class Node:
    def __init__(self, text, doc_id, metadata): self.text, self.doc_id, self.metadata = text, doc_id, metadata

def SentenceSplitter(chunk_size=240, overlap=30):
    def parse(docs):
        nodes = []
        for di, d in enumerate(docs):
            t = re.sub(r"\s+", " ", d.text).strip()
            i = 0
            while i < len(t):
                nodes.append(Node(t[i:i+chunk_size], di, d.metadata))
                i += chunk_size - overlap
        return nodes
    return parse

class Response:
    def __init__(self, response, source_nodes): self.response = response; self.source_nodes = source_nodes
    def __str__(self): return self.response

class QueryEngine:
    def __init__(self, nodes, vecs, llm, top_k=2):
        self.nodes, self.vecs, self.llm, self.top_k = nodes, vecs, llm, top_k
    def query(self, q):
        qv = emb.encode([q], normalize_embeddings=True)[0]
        top = np.argsort(-(self.vecs @ qv))[:self.top_k]
        src = [self.nodes[i] for i in top]
        ctx = "\n".join(f"- {n.text}" for n in src)
        answer = self.llm(f"Context:\n{ctx}\n\nAnswer the question: {q}")
        return Response(answer, src)

class VectorStoreIndex:
    def __init__(self, nodes):
        self.nodes = nodes
        self.vecs = emb.encode([n.text for n in nodes], normalize_embeddings=True)
    @classmethod
    def from_documents(cls, docs, node_parser=None):
        parser = node_parser or SentenceSplitter()
        return cls(parser(docs))
    def as_query_engine(self, llm=None, **kw):
        return QueryEngine(self.nodes, self.vecs, llm or (lambda p: f"[answer using: {p[:60]}...]"), **kw)
    def as_retriever(self, top_k=2):
        def retrieve(q):
            qv = emb.encode([q], normalize_embeddings=True)[0]
            return [self.nodes[i] for i in np.argsort(-(self.vecs @ qv))[:top_k]]
        return retrieve

# the whole LlamaIndex "hello world":
documents = [Document("Refunds are processed within 5 business days to the original card.",
                      {"source": "policy"}),
             Document("Standard shipping is free for orders over fifty dollars.", {"source": "policy"}),
             Document("The REST API allows 600 requests per minute per key.", {"source": "api-docs"})]

index = VectorStoreIndex.from_documents(documents)
qe = index.as_query_engine(llm=lambda p: re.search(r"question: (.*)", p).group(1) + " -> see context",
                           top_k=2)
resp = qe.query("how long do refunds take")
print("answer:", resp)
print("sources:", [n.metadata["source"] for n in resp.source_nodes])

answer: how long do refunds take -> see context
sources: ['policy', 'api-docs']


The point: LlamaIndex optimises the *"I have documents, I want Q&A with citations"* path down
to `from_documents` + `as_query_engine`. It has deep machinery for that path — many index
types (vector, tree, keyword, knowledge-graph), 300+ data connectors (`LlamaHub`), advanced
query engines (sub-question, router, multi-step), and response synthesizers (refine,
tree-summarize, compact).

In [6]:
# LlamaIndex composes too: a retriever from the index feeds a custom step
retrieve = index.as_retriever(top_k=2)
def answer_with_sources(q):
    nodes = retrieve(q)
    return dict(answer=f"[synthesized from {len(nodes)} nodes]",
               sources=[n.metadata.get("source") for n in nodes],
               chunks=[n.text[:50] for n in nodes])
print(json.dumps(answer_with_sources("what is the api rate limit"), indent=1))

{
 "answer": "[synthesized from 2 nodes]",
 "sources": [
  "api-docs",
  "policy"
 ],
 "chunks": [
  "The REST API allows 600 requests per minute per ke",
  "Standard shipping is free for orders over fifty do"
 ]
}


## 4 — The real quickstarts, side by side (10 min)

### LangChain — RAG chain

```python
from langchain_anthropic import ChatAnthropic
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

vs = FAISS.from_texts(texts, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vs.as_retriever(search_kwargs={"k": 4})
prompt = ChatPromptTemplate.from_template(
    "Answer from context only.\n\nContext: {context}\n\nQuestion: {question}")
llm = ChatAnthropic(model="claude-opus-5")

chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt | llm | StrOutputParser())
chain.invoke("how long do refunds take")
```

### LlamaIndex — RAG query engine

```python
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Anthropic(model="claude-opus-5")
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

docs = SimpleDirectoryReader("./knowledge_base").load_data()
index = VectorStoreIndex.from_documents(docs)          # chunk + embed + store
qe = index.as_query_engine(similarity_top_k=4)
resp = qe.query("how long do refunds take")
print(resp, resp.source_nodes)
```

### LangChain — agent with tools

```python
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool

@tool
def get_team(name: str) -> dict:
    "Look up a team's budget and headcount."
    return TEAMS[name]

agent = create_tool_calling_agent(ChatAnthropic(model="claude-opus-5"), [get_team], prompt)
AgentExecutor(agent=agent, tools=[get_team], max_iterations=6).invoke({"input": "..."})
```

## 5 — Choosing (5 min)

| | **LangChain (+ LangGraph)** | **LlamaIndex** | **Neither (roll your own)** |
| --- | --- | --- | --- |
| Sweet spot | general orchestration: agents, tools, multi-step workflows, many integrations | RAG / knowledge assistants: ingestion, indexing, query engines with citations | you understand the pieces and want no dependency churn |
| Core abstraction | `Runnable` / LCEL; LangGraph state machines for agents | `Index` + `QueryEngine` + `Node` | your own functions |
| Retrieval depth | good, generic | **deep** — many index types, response synthesizers, LlamaHub connectors | whatever you build |
| Agent depth | **deep** — LangGraph: branching, checkpoints, human-in-loop | basic agents; leans on function-calling | Day 19 loop + guards |
| Learning/debug cost | high (big surface, fast-moving) | medium | low, but you maintain it |
| Observability | LangSmith (first-party tracing) | callbacks; works with LangSmith/Arize/etc. | you add it (Day 24) |

### Decision guide

1. **RAG over your docs, need citations + swappable retrievers, that's most of the app?**
   → **LlamaIndex.**
2. **An agent with several tools, branching logic, or human approval steps?**
   → **LangGraph** (via LangChain).
3. **A pipeline of prompt → model → parse steps, or you want the integration catalog?**
   → **LangChain / LCEL.**
4. **A focused app you'll maintain for years, you've built the pieces, deps are a liability?**
   → **roll your own** (this course's Weeks 5–7 *are* a hand-rolled framework).
5. **They compose:** LlamaIndex for retrieval as a tool inside a LangGraph agent is common.

Whatever you pick: keep the eval harness (Day 18, Week 9) *outside* the framework so you can
swap frameworks without losing your ground truth.

## 6 — Exercises

1. **Add streaming to `Runnable`.** Give it a `.stream(x)` that yields tokens (split the
   string). Chain `prompt | llm | parser` and stream the final output.
2. **RunnableBranch.** Implement conditional routing: `RunnableBranch((cond, chain_a),
   default=chain_b)`. Route "sentiment" questions to one chain and everything else to another.
3. **Node metadata filter.** Add a `filters=` arg to `as_query_engine` that restricts
   retrieval to nodes whose `metadata["source"]` matches. Verify a query only pulls from
   `api-docs`.
4. **Map your Day 18 pipeline.** Take the `RAGPipeline` class from Day 18 and express its
   ingest + retrieve + generate as (a) an LCEL chain and (b) a LlamaIndex query engine, using
   the mini-frameworks here. Which is fewer lines?
5. **Response synthesizer.** Implement "tree-summarize": if > `top_k` nodes, summarize them
   pairwise up a tree before the final answer. Compare token count to stuffing all nodes.
6. **Framework cost.** List every dependency `pip install langchain langchain-anthropic
   langchain-community llama-index` pulls (use `pip download --no-deps` counts or the docs).
   Argue when that weight is and isn't justified.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. What is a `Runnable` and what does `|` do in LCEL?
2. What are LlamaIndex's two core calls for standard RAG, and what does each hide?
3. Which framework leans retrieval-first and which leans orchestration-first?
4. When does LangChain's docs point you to LangGraph instead?
5. Give two costs a framework adds.
6. You're building a docs Q&A bot with citations and swappable vector stores. Which framework,
   and why not the other?
7. Why keep your eval harness outside whichever framework you choose?

## Where this goes next

- **Day 21 — Build a simple agent:** wire one real tool to a model (Anthropic API, with a
  local fallback) — an agent that decides *when* to search a knowledge base — using the loop
  from Day 19 and, optionally, the framework abstractions from today.